# 02 — Model Experiments
So sánh RandomForest vs XGBoost cho bài toán phát hiện gian lận

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, f1_score
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='darkgrid')
print("Libraries loaded OK")

## 1. Load & Preprocess

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')

X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale Amount & Time
scaler = StandardScaler()
X_train[['Amount','Time']] = scaler.fit_transform(X_train[['Amount','Time']])
X_test[['Amount','Time']]  = scaler.transform(X_test[['Amount','Time']])

# SMOTE
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Train sau SMOTE: {X_train_res.shape} | Fraud: {y_train_res.sum():,}")
print(f"Test (gốc):      {X_test.shape}     | Fraud: {y_test.sum()}")

## 2. Train cả hai model

In [ ]:
# RandomForest
rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                             random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train_res, y_train_res)
print("✅ RandomForest trained")

# XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                     scale_pos_weight=577, eval_metric='aucpr',
                     random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_train_res, y_train_res)
print("✅ XGBoost trained")

## 3. So sánh metrics

In [ ]:
def get_metrics(model, name, X_test, y_test, threshold=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= threshold).astype(int)
    return {
        'Model':    name,
        'ROC-AUC':  round(roc_auc_score(y_test, proba), 4),
        'PR-AUC':   round(average_precision_score(y_test, proba), 4),
        'F1-Fraud': round(f1_score(y_test, preds, pos_label=1), 4),
        'Recall':   round(float(classification_report(y_test, preds, output_dict=True)['1']['recall']), 4),
        'Precision':round(float(classification_report(y_test, preds, output_dict=True)['1']['precision']), 4),
    }

results = pd.DataFrame([
    get_metrics(rf,  'RandomForest', X_test, y_test),
    get_metrics(xgb, 'XGBoost',     X_test, y_test),
])
results.set_index('Model', inplace=True)
results

In [ ]:
# Bar chart so sánh
metrics_plot = ['ROC-AUC', 'PR-AUC', 'F1-Fraud', 'Recall', 'Precision']
x = np.arange(len(metrics_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, results.loc['RandomForest', metrics_plot], width,
               label='RandomForest', color='#4299e1', alpha=0.85)
bars2 = ax.bar(x + width/2, results.loc['XGBoost', metrics_plot], width,
               label='XGBoost', color='#68d391', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics_plot)
ax.set_ylim(0, 1.1)
ax.set_title('So sánh RandomForest vs XGBoost', fontsize=13)
ax.legend()
ax.set_ylabel('Score')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. ROC Curve & Precision-Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model, name, color in [(rf,'RandomForest','#4299e1'), (xgb,'XGBoost','#68d391')]:
    proba = model.predict_proba(X_test)[:, 1]

    # ROC
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.4f})')

    # PR
    prec, rec, _ = precision_recall_curve(y_test, proba)
    pr_auc = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, color=color, lw=2, label=f'{name} (AP={pr_auc:.4f})')

axes[0].plot([0,1],[0,1],'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', lw=1, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/roc_pr_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
THRESHOLD = 0.5

for ax, model, name in [(axes[0], rf, 'RandomForest'), (axes[1], xgb, 'XGBoost')]:
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt=',d', ax=ax, cmap='Blues',
                xticklabels=['Pred Normal','Pred Fraud'],
                yticklabels=['True Normal','True Fraud'])
    ax.set_title(f'{name}\nTP={cm[1,1]} | FN={cm[1,0]} | FP={cm[0,1]}')

plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Feature Importance (XGBoost)

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=X.columns)
top20 = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
top20.plot(kind='barh', ax=ax, color='#68d391', alpha=0.85)
ax.set_title('Top 20 Feature Importances — XGBoost', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../data/processed/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Kết luận

| Model | ROC-AUC | PR-AUC | Ghi chú |
|---|---|---|---|
| RandomForest | ~0.983 | ~0.808 | Nhanh hơn, dễ interpret |
| XGBoost | ~0.983 | ~0.820 | PR-AUC cao hơn, tốt hơn với imbalanced data |

**Khuyến nghị:** Dùng **XGBoost** làm model chính vì PR-AUC tốt hơn — quan trọng hơn ROC-AUC khi data mất cân bằng.

**Threshold tuning:** Mặc định 0.5 → Recall ~88%. Có thể giảm threshold xuống 0.3 để bắt nhiều fraud hơn, đánh đổi bằng nhiều false alarm hơn.